# GRPO Chess Colab Runner

Run the full searchless pipeline in Colab:
1. Pretrain
2. Distill (warm-start from pretrain)
3. GRPO (warm-start from distill)

This notebook creates small smoke-test configs so you can validate end-to-end wiring before expensive runs.


In [ ]:
#@title Runtime Parameters
REPO_URL = "https://github.com/noamdwc/grpo_chess.git"  #@param {type:"string"}
REPO_REF = "feature/lightning_training"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/data/grpo-chess"  #@param {type:"string"}
HF_CACHE_ROOT = "/content/drive/MyDrive/data/grpo-chess/hf_cache"  #@param {type:"string"}
USE_WANDB = True  #@param {type:"boolean"}
WANDB_API_KEY = ""  #@param {type:"string"}
WANDB_PROJECT_PRETRAIN = "chess-grpo-pretrain"  #@param {type:"string"}
WANDB_PROJECT_DISTILL = "chess-grpo-pretrain"  #@param {type:"string"}
WANDB_PROJECT_GRPO = "Chess-GRPO-Bot"  #@param {type:"string"}
RUN_PRETRAIN = True  #@param {type:"boolean"}
RUN_DISTILL = True  #@param {type:"boolean"}
RUN_GRPO = True  #@param {type:"boolean"}


In [ ]:
# Setup workspace
import os
import sys
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

hf_cache_root = Path(HF_CACHE_ROOT if USE_DRIVE else '/content/hf_cache')
hf_cache_root.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(hf_cache_root)
os.environ['HF_DATASETS_CACHE'] = str(hf_cache_root / 'datasets')
os.environ['TRANSFORMERS_CACHE'] = str(hf_cache_root / 'transformers')
print(f"HF cache root: {hf_cache_root}")

if USE_WANDB:
    wandb_key = WANDB_API_KEY.strip()
    if not wandb_key:
        try:
            from google.colab import userdata
            wandb_key = (userdata.get('WANDB_API_KEY') or userdata.get('WANDB_KEY') or '').strip()
        except Exception:
            wandb_key = ''

    if not wandb_key:
        raise RuntimeError(
            'USE_WANDB=True but no key was found. Set WANDB_API_KEY in runtime params or Colab Secrets.'
        )

    os.environ['WANDB_API_KEY'] = wandb_key
    os.environ['WANDB_KEY'] = wandb_key
    print('WandB key configured via environment; logger login happens during training.')
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('WandB disabled for this notebook run.')

!rm -rf /content/grpo_chess
!git clone {REPO_URL} /content/grpo_chess
%cd /content/grpo_chess
!git fetch --all --tags
!git checkout {REPO_REF}
!git submodule update --init --recursive

if '/content/grpo_chess' not in sys.path:
    sys.path.append('/content/grpo_chess')


In [ ]:
# Install dependencies
%pip install -q --upgrade pip
%pip install -q -r requirements.txt
%pip install -q pytorch_lightning wandb python-chess jaxtyping datasets huggingface_hub
!apt-get -qq update
!apt-get -qq install -y stockfish
!which stockfish || true
!python -V


## Create Smoke Configs

These configs are generated inside `src/configs/` and keep runtime/cost low while exercising the full handoff chain.


In [ ]:
from pathlib import Path

repo = Path('/content/grpo_chess')
config_dir = repo / 'src' / 'configs'
config_dir.mkdir(parents=True, exist_ok=True)

artifact_root = Path(DRIVE_ROOT) / 'colab_e2e_smoke' if USE_DRIVE else Path('/content/colab_e2e_smoke')
pretrain_ckpt_dir = artifact_root / 'checkpoints' / 'pretrain'
distill_ckpt_dir = artifact_root / 'checkpoints' / 'distill'
grpo_ckpt_dir = artifact_root / 'checkpoints' / 'grpo'
distill_data_dir = artifact_root / 'distill_data'
pretrain_processed_cache_dir = hf_cache_root / 'pretrain_processed'
hf_datasets_cache_dir = hf_cache_root / 'datasets'

for d in [
    pretrain_ckpt_dir,
    distill_ckpt_dir,
    grpo_ckpt_dir,
    distill_data_dir,
    pretrain_processed_cache_dir,
    hf_datasets_cache_dir,
]:
    d.mkdir(parents=True, exist_ok=True)

pretrain_cfg_name = 'pretrain_colab_e2e_smoke.yaml'
distill_cfg_name = 'distill_colab_e2e_smoke.yaml'
grpo_cfg_name = 'grpo_colab_e2e_smoke.yaml'

wandb_flag = 'true' if USE_WANDB else 'false'
distill_auto_disable_wandb = 'false' if USE_WANDB else 'true'

pretrain_cfg = f"""pretrain:
  lr: 0.0001
  batch_size: 512
  num_epochs: 1
  warmup_steps: 100
  weight_decay: 0.01
  max_grad_norm: 1.0
  checkpoint_dir: "{pretrain_ckpt_dir}"
  resume_from: null
  use_wandb: {wandb_flag}
  wandb_project: "{WANDB_PROJECT_PRETRAIN}"
  label_smoothing: 0.1
  num_workers: 2
  val_check_interval: 1.0
  eval_every_n_epochs: 1000

eval:
  games: 4
  seed: 0
  max_plies: 150
  randomize_opening: true
  opening_plies: 4

stockfish:
  path: "/usr/games/stockfish"
  skill_level: 2
  movetime_ms: 20

policy:
  greedy: true

dataset:
  min_elo: 1800
  max_samples: 20000
  skip_first_n_moves: 5
  skip_last_n_moves: 5
  sample_positions_per_game: 1
  buffer_size: 5000
  filter_abandoned: true
  dataset_name: "Lichess/standard-chess-games"
  split: "train"
  is_eval: false
  eval_fraction: 0.05
  cache_path: "{pretrain_processed_cache_dir}"
  hf_cache_dir: "{hf_datasets_cache_dir}"

transformer:
  vocab_size: 300
  embed_dim: 256
  num_layers: 4
  num_heads: 8
  action_dim: 1968
"""

distill_cfg = f"""generate:
  teacher_model: "9M"
  checkpoint_dir: "searchless_chess/checkpoints"
  checkpoint_step: 6400000
  teacher_batch_size: 256
  top_k: 8
  teacher_temperature: 1.0
  output_dir: "{distill_data_dir}"
  shard_size: 20000
  min_elo: 1800
  max_samples: 50000
  skip_first_n_moves: 5
  skip_last_n_moves: 5
  sample_positions_per_game: 2

deepmind_data:
  num_shards: 1
  top_k: 8
  temperature: 1.0
  min_win_prob: 0.55
  output_dir: "{distill_data_dir}"
  shard_size: 20000

distill:
  lr: 0.0001
  batch_size: 512
  num_epochs: 1
  warmup_steps: 100
  weight_decay: 0.01
  max_grad_norm: 1.0
  checkpoint_dir: "{distill_ckpt_dir}"
  pretrain_checkpoint: "{pretrain_ckpt_dir / 'pretrain_final.pt'}"
  use_wandb: {wandb_flag}
  wandb_project: "{WANDB_PROJECT_DISTILL}"
  log_model_artifacts: false
  auto_disable_wandb_if_missing_key: {distill_auto_disable_wandb}
  fail_on_nonfinite: false
  max_nonfinite_batches: 50
  num_workers: 2
  val_check_interval: 1.0
  eval_every_n_epochs: 1000

eval:
  games: 4
  seed: 0
  max_plies: 150
  randomize_opening: true
  opening_plies: 4

stockfish:
  path: "/usr/games/stockfish"
  skill_level: 2
  movetime_ms: 20

policy:
  greedy: true

dataset:
  data_dir: "{distill_data_dir}"
  top_k: 8
  eval_fraction: 0.05
  max_shards: 1

transformer:
  vocab_size: 300
  embed_dim: 256
  num_layers: 4
  num_heads: 8
  action_dim: 1968
"""

grpo_cfg = f"""training:
  num_epochs: 1
  batch_size: 8
  steps_per_epoch: 32
  checkpoint_dir: "{grpo_ckpt_dir}"
  checkpoint_every_n_epochs: 1
  keep_n_checkpoints: 1
  use_wandb: {wandb_flag}
  wandb_project: "{WANDB_PROJECT_GRPO}"

grpo:
  lr: 0.000001
  num_trajectories: 4
  trajectory_depth: 6
  clip_ratio: 0.20
  kl_coef: 0.001
  eval_every_n_epochs: 1000
  ppo_steps: 1
  rollout_temperature: 1.2
  enable_safety_checks: false
  safety_patience_steps: 1000
  max_clip_fraction: 0.95
  teacher_forcing_prob: 0.0
  teacher_forcing_depth: 4

transformer:
  vocab_size: 300
  embed_dim: 256
  num_layers: 4
  num_heads: 8
  action_dim: 1968

eval:
  games: 4
  seed: 0
  max_plies: 150
  randomize_opening: true
  opening_plies: 4

stockfish:
  path: "/usr/games/stockfish"
  skill_level: 2
  use_elo_limit: false
  elo: 2500
  movetime_ms: 20
  threads: 1
  hash_mb: 64

policy:
  temperature: 0.8
  greedy: true
  branching_factor: 4
  search_depth: 2

searcher: null

pretrain:
  checkpoint_path: "{distill_ckpt_dir / 'distill_final.pt'}"
  freeze_layers: 1

dataset:
  max_steps: 32
  phase_distribution:
    opening: 0.33
    middlegame: 0.34
    endgame: 0.33
  min_eval_cp: -200
  max_eval_cp: 200
  quality_filter: true
  stockfish_filter_depth: 4
"""

(config_dir / pretrain_cfg_name).write_text(pretrain_cfg)
(config_dir / distill_cfg_name).write_text(distill_cfg)
(config_dir / grpo_cfg_name).write_text(grpo_cfg)

print('Wrote configs:')
print('-', pretrain_cfg_name)
print('-', distill_cfg_name)
print('-', grpo_cfg_name)
print('Artifact root:', artifact_root)
print('HF datasets cache:', hf_datasets_cache_dir)


## Run Pipeline Stages


In [ ]:
# Stage 1: Pretrain
if RUN_PRETRAIN:
    !python -m src.pretrain.pretrain --config pretrain_colab_e2e_smoke.yaml
else:
    print('Skipping pretrain stage')


In [ ]:
# Stage 2: Distill (warm-start from pretrain_final.pt)
if RUN_DISTILL:
    !python -m src.distill.distill --config distill_colab_e2e_smoke.yaml
else:
    print('Skipping distill stage')


In [ ]:
# Stage 3: GRPO (warm-start from distill_final.pt)
if RUN_GRPO:
    !python -m src.train_self_play --config grpo_colab_e2e_smoke.yaml
else:
    print('Skipping GRPO stage')


In [ ]:
# Verify expected artifacts
from pathlib import Path

checks = {
    'pretrain_final': pretrain_ckpt_dir / 'pretrain_final.pt',
    'distill_final': distill_ckpt_dir / 'distill_final.pt',
    'grpo_dir': grpo_ckpt_dir,
}
for name, path in checks.items():
    print(f'{name}:', 'OK' if path.exists() else f'MISSING ({path})')


## Notes

- For full training runs, increase epochs/steps in the generated config cell.
- This project is searchless; this notebook does not introduce tree-search/MCTS steps.
